# CoWork Enterprise Demo — Dev → Prod Promotion
## Notebook 06 — Dev → Prod Promotion (Agent-as-Code)

_Icons used throughout: 🛠️ Action  📌 Note  🔹 Info_

> ⚠️ _Generated from `build_notebooks.py` — edit the builder and re-run. Direct edits to this notebook are overwritten._

---

### What This Notebook Does:

1. 🛠️ **Captures** the agent spec (the source of truth) with `DESCRIBE AGENT`
2. 🛠️ **Versions** it in place with `ALTER AGENT ... MODIFY LIVE VERSION`
3. 🛠️ **Deploys** the identical spec to a prod-scoped target (`AGENTS_PROD`)
4. 🛠️ **Gates** promotion on an evaluation pass, then **publishes** and cuts over

---

### Why Promote the Spec, Not the Running Agent:

🔹 The CoWork object is a **per-account singleton**, so real enterprises run **dev and prod in separate accounts** (or at least separate databases/schemas). You don't promote a *running* agent — you promote its **specification**.

🔹 The agent is created `FROM SPECIFICATION`; that YAML spec is your **source of truth** (keep it in git). **Promotion = deploy the same spec to the prod target, gated by an evaluation pass.** Never hand-edit a live prod agent — edit the spec in source control and redeploy.

This notebook shows the mechanics on one account using a prod-scoped schema (`AGENTS_PROD`); the same steps apply across accounts by pointing the deploy at the prod account.

**Promotion model:** capture spec → version in place → deploy to prod → evaluate (gate) → publish.

---

### Estimated Time: **20–25 minutes**

### Prerequisites:
- Notebooks 00–05 complete (they build the dev agent `DEMO_AGENT`)

In [ ]:
%%sql -r set_context
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
USE DATABASE COWORK_ENTERPRISE_DEMO;
USE SCHEMA AGENTS;
USE WAREHOUSE COWORK_ENTERPRISE_DEMO_WH;


## Step 1: Agent-as-Code — Capture the Spec (source of truth)
`DESCRIBE AGENT` returns the full spec in the `agent_spec` column - this is the code you version in git. `DESCRIBE AS RESOURCE AGENT <name>` returns the same as JSON. Never hand-edit prod: edit the spec in source control and redeploy.

In [ ]:
%%sql -r capture_spec
DESCRIBE AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT;


## Step 2: Version in Place (iterate safely)
Agents are versioned. `ALTER AGENT ... MODIFY LIVE VERSION SET SPECIFICATION = $$...$$` replaces the LIVE version's spec in one atomic step (fields omitted are removed). Tag each promoted build with a COMMENT so you can trace what shipped.

```sql
-- Replace the LIVE version's spec (whole-spec replacement):
ALTER AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT MODIFY LIVE VERSION SET SPECIFICATION = $$ ...updated spec... $$;
```

In [ ]:
%%sql -r tag_version
ALTER AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT SET COMMENT = 'v1 - passed evaluation; promoted to prod';


## Step 3: Deploy to the Prod Target (same spec)
Create the prod-scoped schema and deploy **the identical spec** with `CREATE OR REPLACE AGENT ... FROM SPECIFICATION`. In a real cross-account promotion this runs in the prod account and the `tool_resources` point at prod data; here we reuse the demo objects to show the mechanics.

In [ ]:
%%sql -r create_prod_schema
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
CREATE SCHEMA IF NOT EXISTS COWORK_ENTERPRISE_DEMO.AGENTS_PROD
  COMMENT = 'Promotion target (prod-scoped) for the CoWork enterprise demo';


In [ ]:
%%sql -r promote_agent
CREATE OR REPLACE AGENT COWORK_ENTERPRISE_DEMO.AGENTS_PROD.DEMO_AGENT
  COMMENT = 'DEMO_AGENT promoted from dev - identical specification'
  PROFILE = '{"display_name": "Enterprise Demo Analyst", "color": "blue"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 45
    tokens: 16000

instructions:
  response: |
    You are the Enterprise Demo Analyst. You help portfolio managers, relationship managers,
    and compliance officers understand client portfolios, trading activity, and market research.

    Guidelines:
    - Be concise and data-driven. Lead with numbers when available.
    - When showing financial data, format large numbers (e.g., $2.5B not $2500000000).
    - If a question spans both structured data (portfolios, trades) and unstructured data
      (research notes), use BOTH tools to provide a complete answer.
    - Always cite which data source your answer came from.
    - For compliance questions, flag any potential issues clearly.
  orchestration: |
    - For questions about client AUM, portfolio values, positions, or trades: use the Analyst tool.
    - For questions about market outlook, research opinions, or compliance reports: use the Search tool.
    - For questions that need both, use both tools.
  sample_questions:
    - question: "Who are our top 5 clients by AUM?"
    - question: "What is our total technology sector exposure?"
    - question: "Are there any compliance concerns I should know about?"
    - question: "Show me recent trades over $1M and the rationale behind them."

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "nexus_analytics"
      description: "Query structured financial data including client accounts, portfolio positions, trade history, and business metrics. Use this for any question about numbers, rankings, aggregations, or trends."
  - tool_spec:
      type: "cortex_search"
      name: "nexus_research"
      description: "Search analyst research notes, market commentary, risk assessments, and compliance reports. Use this for qualitative insights, opinions, recommendations, and compliance flags."
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Generate visualizations from query results when the user asks for charts or visual breakdowns."

tool_resources:
  nexus_analytics:
    semantic_view: "COWORK_ENTERPRISE_DEMO.SEMANTIC.DEMO_SV"
    execution_environment:
      type: warehouse
      warehouse: "COWORK_ENTERPRISE_DEMO_WH"
  nexus_research:
    name: "COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_SEARCH"
    max_results: "5"
$$;
SELECT 'Agent promoted to AGENTS_PROD.DEMO_AGENT' AS STATUS;


In [ ]:
%%sql -r grant_prod_usage
USE ROLE ACCOUNTADMIN;
GRANT USAGE ON AGENT COWORK_ENTERPRISE_DEMO.AGENTS_PROD.DEMO_AGENT TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
SELECT 'USAGE granted on promoted agent' AS STATUS;


## Step 4: Promotion Gate = Evaluation Pass
Only promote a build that clears the evaluation threshold (Notebook 05's `EXECUTE_AI_EVALUATION` on the golden-question bank). Here we smoke-test the promoted agent to confirm it answers before we publish it. **If it fails the gate, do not publish - fix the spec in git and redeploy.**

In [ ]:
%%sql -r gate_eval
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'COWORK_ENTERPRISE_DEMO.AGENTS_PROD.DEMO_AGENT',
  '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Who are our top 5 clients by AUM?"}]}], "stream": false}'
) AS resp;


## Step 5: Publish the Promoted Agent to CoWork (cut over)
Add the **promoted** agent to the CoWork object. To cut over cleanly, remove the dev agent so users see only the promoted build. Requires `MODIFY` on the object, so we use ACCOUNTADMIN.

In [ ]:
%%sql -r publish_prod
USE ROLE ACCOUNTADMIN;
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT ADD AGENT COWORK_ENTERPRISE_DEMO.AGENTS_PROD.DEMO_AGENT;
-- Cut over: remove the dev build so only the promoted agent is visible.
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT DROP AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT;


In [ ]:
%%sql -r verify_prod_published
SHOW AGENTS IN SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT;


## Environment strategy (reference)

| Concern | Pattern |
| --- | --- |
| Environments | Separate **accounts** (or DBs) for dev / prod - the CoWork object is one per account |
| Source of truth | The `FROM SPECIFICATION` spec, checked into **git** (agent-as-code) |
| Deploy | `CREATE OR REPLACE AGENT ... FROM SPECIFICATION` (or `ALTER AGENT MODIFY LIVE VERSION`) |
| Promotion gate | Evaluation threshold (Notebook 05) must pass **before** publish |
| Cut over | Publish the promoted agent to the CoWork object; remove the prior build |
| Change safety | Clone-before-change on prod; never hand-edit a live agent |
| CI/CD | Script the deploy in a task / pipeline; version + review the spec like any code |

## ✅ Notebook 06 Complete

### What You Just Built:
- The dev agent spec captured as code and versioned with a traceable COMMENT
- The identical spec deployed to `COWORK_ENTERPRISE_DEMO.AGENTS_PROD.DEMO_AGENT` and smoke-tested through the gate
- The promoted agent published to CoWork with a clean cut-over from the dev build

---

### Key Points:
- You promote the spec, not the running agent — the spec is the git source of truth.
- The evaluation threshold is the promotion gate: no pass, no publish.
- The CoWork object is one per account, so dev/prod belong in separate accounts (or schemas).

---

### Next:

Continue to **Notebook 07 — Continuous Improvement**: mine real usage, re-evaluate, and optimize the agent's instructions in a governed loop. (Run **Notebook 08** last to clean up.)